In [1]:
# saving and loading the model

In [2]:
import pickle

In [ ]:
output_file = 'model_C=%s.bin' % C

In [ ]:
f_out = open(output_file, 'wb')
pickle.dump((dv, model, f_out))
f_out.close()

In [ ]:
with open(output_file, 'wb') as f_out:
    pickle.dump(dv, model, f_out)

In [ ]:
import pickle

In [ ]:
with open(output_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [ ]:
dv, model

In [ ]:
customer = {
    '':''
}

In [ ]:
X = dv.transform([customer])

In [ ]:
model.predict_proba(X)[0, 1]

In [1]:
# Introduction Flask

In [ ]:
from flask import Flask

In [ ]:
app = Flask('ping')

In [ ]:
@app.route('/ping', methods=['GET'])
def ping():
    return "PONG"

In [ ]:
if __name__ ==  "__main__":
app.run(debug=True, host='0.0.0.0', port=9696)

In [1]:
# Serving the churn model with Flask

In [ ]:
@app.route('/predict', methods=['POST'[)
def predict(customer):
    customer = request.get_json()
    X = dv.transform([customer])
    y_pred = model.predict_proba(X)[0, 1]
    churn = y_pred >= 0.5

    result = {
        'churn_probability': y_pred,
        'churn': churn
    }
    return result

In [2]:
import requests

In [ ]:
url = 'http://localhost:9696/predict'

In [ ]:
customer = {
    '':''
}

In [ ]:
response = requests.post(url, json=customer).json()
response

In [ ]:
if response['churn'] == True:
    print('sending promo email')

In [1]:
# Pivenv

In [ ]:
pip install pipenv

In [ ]:
pipenv install numpy scikit-learn==0.24.2 flask

In [ ]:
pipenv install gunicorn

In [ ]:
pipenv shell

In [ ]:
pipenv run gunicorn --bind 0.0.0.0:9696 predict:app

In [1]:
# docker

In [ ]:
sudo apt-get install docker.io

In [ ]:
# First install the python 3.8, the slim version uses less space
FROM python:3.8.12-slim

# Install pipenv library in Docker 
RUN pip install pipenv

# create a directory in Docker named app and we're using it as work directory 
WORKDIR /app                                                                

# Copy the Pip files into our working derectory 
COPY ["Pipfile", "Pipfile.lock", "./"]

# install the pipenv dependencies for the project and deploy them.
RUN pipenv install --deploy --system

# Copy any python files and the model we had to the working directory of Docker 
COPY ["*.py", "churn-model.bin", "./"]

# We need to expose the 9696 port because we're not able to communicate with Docker outside it
EXPOSE 9696

# If we run the Docker image, we want our churn app to be running
ENTRYPOINT ["gunicorn", "--bind=0.0.0.0:9696", "churn_serving:app"]

In [1]:
# AWS

In [ ]:
# Install the AWS EB CLI
pipenv install awsebcli --dev

In [ ]:
# Initialize the EB environment
eb init -p docker -r eu-north-1 churn-serving

In [ ]:
# Verify the EB Configuration
less .elasticbeanstalk/config.yml

In [ ]:
# Test locally
eb local run --port 9696

In [ ]:
# Deploy to AWS
eb create churn-serving-env